In [3]:
## 🔹 步骤
#
# 1. 下载 `retail.dat`（SPMF 官方提供）。
# 2. 加载数据并转为事务列表。
# 3. 用 `mlxtend.preprocessing.TransactionEncoder` 转成 one-hot 矩阵。
# 4. 用 `mlxtend.frequent_patterns.fpgrowth` 挖掘频繁项集。
# 5. 用 `mlxtend.frequent_patterns.association_rules` 生成规则。
#
# ---

## 🔹 代码示例

import os
import urllib.request
from itertools import combinations

import numpy as np
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

# 1. 下载 retail.dat
# url = "https://www.philippe-fournier-viger.com/spmf/datasets/retail.txt"
# file_path = "retail.txt"
#
# if not os.path.exists(file_path):
#     print("正在下载 retail.txt ...")
#     urllib.request.urlretrieve(url, file_path)
#     print("下载完成！")


## retail.txt 就是一个 “购物篮数据集”，每行表示一位顾客的一次购买记录，用商品编号代替实际商品名称，常用于关联规则算法的测试与比较。

In [34]:

# 2. 加载数据
transactions = []
with open("./retail.txt", "r") as f:
    for line in f:
        items = list(map(int, line.strip().split()))
        transactions.append(items)

print(f"交易数: {len(transactions)}")
print(f"示例交易: {transactions[0]}")

# ⚠️ 采样（可选），否则数据太大挖掘会比较慢
sample_transactions = transactions[:10000]

# 3. 转换为 One-Hot 矩阵
te = TransactionEncoder()
te_ary = te.fit(sample_transactions).transform(sample_transactions)
df = pd.DataFrame(te_ary, columns=te.columns_)

print("One-Hot 矩阵维度:", df.shape)
print("One-Hot values:", df.head(10))

# 4. FP-Growth 挖掘频繁项集
frequent_itemsets = fpgrowth(df, min_support=0.02, use_colnames=True)  # 支持度阈值可调整
print("\n频繁项集示例:")
print(frequent_itemsets.head())

# 5. 生成关联规则
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
print("\n关联规则示例:")
print(rules.head())

交易数: 88162
示例交易: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
One-Hot 矩阵维度: (10000, 8600)
One-Hot values:     1      2      3      4      5      6      7      8      9      10    ...  \
0   True   True   True   True   True   True   True   True   True   True  ...   
1  False  False  False  False  False  False  False  False  False  False  ...   
2  False  False  False  False  False  False  False  False  False  False  ...   
3  False  False  False  False  False  False  False  False  False  False  ...   
4  False  False  False  False  False  False  False  False  False  False  ...   
5  False  False  False  False  False  False  False  False  False  False  ...   
6  False  False  False  False  False  False  False  False  False  False  ...   
7  False  False  False   True  False  False  False  False  False  False  ...   
8  False  False  False  False  False  False  False  False  False  False  ...   
9  False  False  False  Fal

In [5]:
sample_transactions = transactions[:10000]

In [10]:
type(sample_transactions[:1])

list

In [15]:
sample_transactions[:10]

[[1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30],
 [31, 32, 33],
 [34, 35, 36],
 [37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47],
 [39, 40, 48, 49],
 [39, 40, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59],
 [33, 42, 60, 61, 62, 63],
 [4, 40, 49],
 [64, 65, 66, 67, 68, 69],
 [33, 70]]

In [39]:
import pandas as pd
import numpy as np
retail_type = set()
for transaction in sample_transactions:
    retail_type.update(transaction)
retail_one_hot = np.zeros((len(sample_transactions), len(retail_type) + 1))
for idx, transaction in enumerate(sample_transactions):
    retail_one_hot[idx, transaction] = 1

retail_one_hot = retail_one_hot[:, 1:]

In [40]:
retail_one_hot.astype(bool)
np.sum(retail_one_hot == te_ary)
## 说明转换效果等价

np.int64(86000000)

In [45]:
retail_one_hot = retail_one_hot.astype(int)
retail_one_hot[:10, :]

array([[1, 1, 1, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [74]:
from mlxtend.frequent_patterns import apriori

## apriori只支持True/False的pandas二位数组
retail_one_hot = retail_one_hot.astype(bool)
df_retail_one_hot = pd.DataFrame(retail_one_hot, columns=np.arange(1, retail_one_hot.shape[1] + 1))
## 这个apriori算法 非常耗时,记录数、列数一定要控制尽量少
apriori = apriori(df_retail_one_hot.iloc[:100, :10], min_support=0.05, use_colnames=True)
apriori

,support,itemsets
0,0.1,(1)
1,0.1,(2)
2,0.1,(3)
3,0.2,(4)
4,0.1,(5)
...,...,...
1018,0.1,"(1, 2, 3, 5, 6, 7, 8, 9, 10)"
1019,0.1,"(1, 2, 4, 5, 6, 7, 8, 9, 10)"
1020,0.1,"(1, 3, 4, 5, 6, 7, 8, 9, 10)"
1021,0.1,"(2, 3, 4, 5, 6, 7, 8, 9, 10)"


In [72]:
df_retail_one_hot.iloc[:10, :10]

,1,2,3,4,5,6,7,8,9,10
0,True,True,True,True,True,True,True,True,True,True
1,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False
5,False,False,False,False,False,False,False,False,False,False
6,False,False,False,False,False,False,False,False,False,False
7,False,False,False,True,False,False,False,False,False,False
8,False,False,False,False,False,False,False,False,False,False
9,False,False,False,False,False,False,False,False,False,False


In [64]:
data = {
    'ID':[1,2,3,4,5,6],
    'Onion':[1,0,0,1,1,1],
    'Potato': [1,1,0,1,1,1],
    'Burger': [1,1,0,0,1,1],
    'Milk': [0,1,1,1,0,1],
    'Beer': [0,0,1,0,1,0],
}


In [81]:
import pandas as pd
df = pd.DataFrame(data)
df_data = df.iloc[:, 1:].astype(bool)
df_data

In [84]:
from mlxtend.frequent_patterns import apriori

##  计算支持度 -- 来选择频繁项集
support = apriori(df_data, min_support=0.5, use_colnames=True)
support.sort_values(by=['support'], ascending=False, inplace=True)

In [85]:
support

,support,itemsets
1,0.833333,(Potato)
0,0.666667,(Onion)
2,0.666667,(Burger)
3,0.666667,(Milk)
4,0.666667,"(Potato, Onion)"
6,0.666667,"(Potato, Burger)"
5,0.500000,"(Onion, Burger)"
7,0.500000,"(Potato, Milk)"
8,0.500000,"(Potato, Onion, Burger)"


In [90]:
##  计算关联规则 -- 指定评估方法 lift--是否独立
from mlxtend.frequent_patterns import association_rules
rules = association_rules(support, metric="lift", min_threshold=1)

In [94]:
rules[ (rules['lift'] > 1.125) & (rules['confidence'] >= 0.8) ]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
1,(Onion),(Potato),0.666667,0.833333,0.666667,1.0,1.2,1.0,0.111111,inf,0.500000,0.8,1.0,0.9
3,(Burger),(Potato),0.666667,0.833333,0.666667,1.0,1.2,1.0,0.111111,inf,0.500000,0.8,1.0,0.9
8,"(Onion, Burger)",(Potato),0.500000,0.833333,0.500000,1.0,1.2,1.0,0.083333,inf,0.333333,0.6,1.0,0.8


结论:
1. Onion和Potato, Burger和Potato 可以搭配卖
2. 如果Onion和Burger已经购买了, 可以询问顾客是否需要土豆, 因为购买了前两样的物品,购买土豆的概率高

In [99]:
retail_shopping_basket = {
    'ID':[1,2,3,4,5,6],
    'Basket' : [['Beer', 'Diaper', 'Pretzels', 'Chips', 'Aspirin'],
         ['Diaper', 'Beer', 'Chips', 'Lotion', 'Juice', 'BabyFood', 'Milk'],
         ['Soda', 'Chips', 'Milk'],
         ['Soup', 'Beer', 'Diaper', 'Milk', 'IceCream'],
         ['Soda', 'Coffee', 'Milk', 'Bread'],
         ['Beer', 'Chips']
        ]
}

In [112]:
retail = pd.DataFrame(retail_shopping_basket)

In [113]:
retail

,ID,Basket
0,1,"[Beer, Diaper, Pretzels, Chips, Aspirin]"
1,2,"[Diaper, Beer, Chips, Lotion, Juice, BabyFood, Milk]"
2,3,"[Soda, Chips, Milk]"
3,4,"[Soup, Beer, Diaper, Milk, IceCream]"
4,5,"[Soda, Coffee, Milk, Bread]"
5,6,"[Beer, Chips]"


In [115]:
col = retail['Basket'].str.join(',')

In [116]:
col

0              Beer,Diaper,Pretzels,Chips,Aspirin
1    Diaper,Beer,Chips,Lotion,Juice,BabyFood,Milk
2                                 Soda,Chips,Milk
3                  Soup,Beer,Diaper,Milk,IceCream
4                          Soda,Coffee,Milk,Bread
5                                      Beer,Chips
Name: Basket, dtype: object

In [120]:
retail_oh = col.str.get_dummies(',')
retail_oh

,Aspirin,BabyFood,Beer,Bread,Chips,Coffee,Diaper,IceCream,Juice,Lotion,Milk,Pretzels,Soda,Soup
0,1,0,1,0,1,0,1,0,0,0,0,1,0,0
1,0,1,1,0,1,0,1,0,1,1,1,0,0,0
2,0,0,0,0,1,0,0,0,0,0,1,0,1,0
3,0,0,1,0,0,0,1,1,0,0,1,0,0,1
4,0,0,0,1,0,1,0,0,0,0,1,0,1,0
5,0,0,1,0,1,0,0,0,0,0,0,0,0,0


In [124]:
res = pd.concat([retail['ID'], retail_oh], axis=1)
res.set_index('ID')
res

,ID,Aspirin,BabyFood,Beer,Bread,Chips,Coffee,Diaper,IceCream,Juice,Lotion,Milk,Pretzels,Soda,Soup
0,1,1,0,1,0,1,0,1,0,0,0,0,1,0,0
1,2,0,1,1,0,1,0,1,0,1,1,1,0,0,0
2,3,0,0,0,0,1,0,0,0,0,0,1,0,1,0
3,4,0,0,1,0,0,0,1,1,0,0,1,0,0,1
4,5,0,0,0,1,0,1,0,0,0,0,1,0,1,0
5,6,0,0,1,0,1,0,0,0,0,0,0,0,0,0


In [128]:
from mlxtend.frequent_patterns import apriori
retail_oh = retail_oh.astype(bool)
frequent_items = apriori(retail_oh, min_support=0.5, use_colnames=True)
frequent_items

,support,itemsets
0,0.666667,(Beer)
1,0.666667,(Chips)
2,0.500000,(Diaper)
3,0.666667,(Milk)
4,0.500000,"(Beer, Chips)"
5,0.500000,"(Diaper, Beer)"


In [129]:
from mlxtend.frequent_patterns import association_rules
rules = association_rules(frequent_items, metric="lift", min_threshold=1)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(Beer),(Chips),0.666667,0.666667,0.5,0.75,1.125,1.0,0.055556,1.333333,0.333333,0.60,0.25,0.750
1,(Chips),(Beer),0.666667,0.666667,0.5,0.75,1.125,1.0,0.055556,1.333333,0.333333,0.60,0.25,0.750
2,(Diaper),(Beer),0.500000,0.666667,0.5,1.00,1.500,1.0,0.166667,inf,0.666667,0.75,1.00,0.875
3,(Beer),(Diaper),0.666667,0.500000,0.5,0.75,1.500,1.0,0.166667,2.000000,1.000000,0.75,0.50,0.875


# Apriori算法的实现
- Apriori定律1 ：如果某商品组合小于最小支持度，则就将它舍去，它的超集必然不是频繁项集。
- Apriori定律2 ：如果一个集合是频繁项集，即这个商品组合支持度大于最小支持度，则它的所有子集都是频繁项集

In [160]:
from itertools import combinations

import numpy as np

class MyApriori:
    def __init__(self, data, min_support = 0.5):
        self.data = data
        self.min_support = min_support
        self.data_one_hot, self.item_index_dict = self.to_one_hot(data)

        result = []
        candidate_items = list(self.item_index_dict.keys())
        candidate_items.sort()
        while len(candidate_items) >= 1:
            frequent_item_dict, frequent_item_keys = self.scan(candidate_items) #频繁项集
            if frequent_item_dict:
                result.append(frequent_item_dict)
                candidate_items = self.get_candidate_items(frequent_item_keys)  # 候选集

        support = {}
        for res_dict in result:
            for k, v in res_dict.items():
                support[k] = v
        self.support = support

    def get_candidate_items(self, frequent_item_keys):
        candidate_items = []
        if len(frequent_item_keys) > 1:
            frequent_items_len = len(frequent_item_keys)
            for i in range(frequent_items_len):
                for j in range(i + 1, frequent_items_len):
                    if self.hava_same_pre_candidate_key(frequent_item_keys[i], frequent_item_keys[j]):
                        candidate_items.append(frequent_item_keys[i] + "," + frequent_item_keys[j].split(",")[-1])
        return candidate_items

    def hava_same_pre_candidate_key(self, frequent_items_prev, frequent_items_next):
        prefix_prev = ",".join(frequent_items_prev.split(",")[:-1])
        prefix_next = ",".join(frequent_items_next.split(",")[:-1])
        return prefix_prev == prefix_next

    def scan(self, candidate_items):
        freq_item_dict = {}
        freq_item_keys = []
        for item in candidate_items:
            indexs = self.to_index(item)
            count = 0
            for row in self.data_one_hot:
                if np.sum(row[indexs]) == len(indexs):
                    count += 1
            support = count / len(self.data_one_hot)
            if support >= self.min_support:
                freq_item_dict[item] = support
                freq_item_keys.append(item)
        return freq_item_dict, freq_item_keys

    def to_index(self, item):
        indexs = []
        keys = item.split(",")
        for key in keys:
            indexs.append(self.item_index_dict[key.strip()])
        return indexs

    def to_one_hot(self, data):
        item_type = set()
        for items in data:
            item_type.update(items)
        item_type = list(item_type)
        item_type.sort()
        item_index_dict = {item: i for i, item in enumerate(item_type)}
        one_hot = np.zeros((len(data), len(item_type)), dtype=int)
        for idx, items in enumerate(data):
            for item in items:
                one_hot[idx, item_index_dict[item]] = 1
        return one_hot, item_index_dict

    def generate_rules(self, freq_itemsets, min_conf=0.6):
        """
        freq_itemsets: DataFrame, 包含 [support, itemsets]
        min_conf: 最小置信度
        """
        rules = []
        for itemset, supp in freq_itemsets.items():
            itemset_arr = itemset.split(",")
            sorted(itemset_arr)
            if len(itemset_arr) < 2:
                continue
            # 遍历所有可能的划分
            for i in range(1, len(itemset_arr)):
                for left in combinations(itemset_arr, i):
                    left = sorted(left)
                    right = sorted(set(itemset_arr) - set(left))
                    supp_xy = freq_itemsets[itemset]
                    supp_x = freq_itemsets[",".join(left)]
                    supp_y = freq_itemsets[",".join(right)]
                    conf = supp_xy / supp_x
                    lift = conf / supp_y
                    if conf >= min_conf:
                        rules.append({
                            "X": left,
                            "Y": right,
                            "support": supp_xy,
                            "confidence": conf,
                            "lift": lift
                        })
        return rules


In [161]:

data = {
    "Tid": [1, 2, 3, 4],
    "Items" : [
        ["A", "C", "D"],
        ["B", "C", "E"],
        ["A", "B", "C", "E"],
        ["B", "E"],
    ]
}
my_apriori = MyApriori(data["Items"], min_support = 0.5)
print(my_apriori.support)

{'A': 0.5, 'B': 0.75, 'C': 0.75, 'E': 0.75, 'A,C': 0.5, 'B,C': 0.5, 'B,E': 0.75, 'C,E': 0.5, 'B,C,E': 0.5}


In [164]:
rules = my_apriori.generate_rules(my_apriori.support)
rules_dict = {}
for rule in rules:
    X, Y, support, confidence, lift = [],[],[],[],[]
    for rule in rules:
        X.append(rule["X"])
        Y.append(rule["Y"])
        support.append(rule["support"])
        confidence.append(rule["confidence"])
        lift.append(rule["lift"])
    rules_dict["X"] = X
    rules_dict["Y"] = Y
    rules_dict["support"] = support
    rules_dict["confidence"] = confidence
    rules_dict["lift"] = lift

pd.DataFrame.from_dict(rules_dict)


,X,Y,support,confidence,lift
0,[A],[C],0.50,1.000000,1.333333
1,[C],[A],0.50,0.666667,1.333333
2,[B],[C],0.50,0.666667,0.888889
3,[C],[B],0.50,0.666667,0.888889
4,[B],[E],0.75,1.000000,1.333333
5,[E],[B],0.75,1.000000,1.333333
6,[C],[E],0.50,0.666667,0.888889
7,[E],[C],0.50,0.666667,0.888889
8,[B],"[C, E]",0.50,0.666667,1.333333
9,[C],"[B, E]",0.50,0.666667,0.888889


In [146]:
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules
new_data = pd.DataFrame(data, columns=["Tid", "Items"])
new_data = new_data['Items'].str.join(',')
new_data = new_data.str.get_dummies(',').astype(bool)
apriori_2 = apriori(new_data, min_support=0.5, use_colnames=True)
print(apriori_2)
rules = association_rules(apriori_2, metric="lift", min_threshold=1)
rules

   support   itemsets
0     0.50        (A)
1     0.75        (B)
2     0.75        (C)
3     0.75        (E)
4     0.50     (C, A)
5     0.50     (B, C)
6     0.75     (E, B)
7     0.50     (E, C)
8     0.50  (E, B, C)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(C),(A),0.75,0.50,0.50,0.666667,1.333333,1.0,0.1250,1.5,1.0,0.666667,0.333333,0.833333
1,(A),(C),0.50,0.75,0.50,1.000000,1.333333,1.0,0.1250,inf,0.5,0.666667,1.000000,0.833333
2,(E),(B),0.75,0.75,0.75,1.000000,1.333333,1.0,0.1875,inf,1.0,1.000000,1.000000,1.000000
3,(B),(E),0.75,0.75,0.75,1.000000,1.333333,1.0,0.1875,inf,1.0,1.000000,1.000000,1.000000
4,"(E, C)",(B),0.50,0.75,0.50,1.000000,1.333333,1.0,0.1250,inf,0.5,0.666667,1.000000,0.833333
5,"(B, C)",(E),0.50,0.75,0.50,1.000000,1.333333,1.0,0.1250,inf,0.5,0.666667,1.000000,0.833333
6,(E),"(B, C)",0.75,0.50,0.50,0.666667,1.333333,1.0,0.1250,1.5,1.0,0.666667,0.333333,0.833333
7,(B),"(E, C)",0.75,0.50,0.50,0.666667,1.333333,1.0,0.1250,1.5,1.0,0.666667,0.333333,0.833333


In [156]:
def get_all_sub_seq(items, r):
    length = len(items)
    if r > length:
        return

    sub_seqs = []
    indices = list(range(r)) # 0 1
    sub_seqs.append(get_sub(items, indices))
    while True:
        pos = compute_pos(items, indices) ## 计算可推进位置
        if pos is None:
            break
        ## 更新下标
        indices[pos] = indices[pos] + 1
        while pos < (r - 1):
            indices[pos + 1] = indices[pos] + 1
            pos += 1
        sub_seqs.append(get_sub(items, indices))
    return sub_seqs

def compute_pos(items, indices):
    length = len(items)
    indices_len = len(indices)
    for i in range(indices_len):
        if indices[indices_len -i - 1] < (length-i-1):
            return indices_len -i - 1
    return None

def get_sub(items, indices):
    return tuple([items[i] for i in indices])


print(get_all_sub_seq(['A', 'B', 'C', 'D', 'E'], 3))

[('A', 'B', 'C'), ('A', 'B', 'D'), ('A', 'B', 'E'), ('A', 'C', 'D'), ('A', 'C', 'E'), ('A', 'D', 'E'), ('B', 'C', 'D'), ('B', 'C', 'E'), ('B', 'D', 'E'), ('C', 'D', 'E')]


In [148]:
from itertools import combinations
items = ['A', 'B', 'C', 'D', 'E']
combinations = list(combinations(items, 3))
combinations

[('A', 'B', 'C'),
 ('A', 'B', 'D'),
 ('A', 'B', 'E'),
 ('A', 'C', 'D'),
 ('A', 'C', 'E'),
 ('A', 'D', 'E'),
 ('B', 'C', 'D'),
 ('B', 'C', 'E'),
 ('B', 'D', 'E'),
 ('C', 'D', 'E')]

In [152]:
tuple(items)[0, 1]

'A'

In [174]:
def get_subsequences(collection, index, current, result):
    # 如果当前子集非空且不是全集，添加到结果
    if current and len(current) < len(collection):
        result.append(','.join(current))
    # 遍历从 index 开始的剩余元素
    for i in range(index, len(collection)):
        # 包含当前元素，继续递归
        print(f"get_subsequences({collection}, {i + 1}, {current + [collection[i]]})")
        get_subsequences(collection, i + 1, current + [collection[i]], result)
        print("-------------")

# 示例
collection = ['A', 'B', 'C']
result = []
get_subsequences(collection, 0, [], result)
# 按逗号拼接结果
result

get_subsequences(['A', 'B', 'C'], 1, ['A'])
get_subsequences(['A', 'B', 'C'], 2, ['A', 'B'])
get_subsequences(['A', 'B', 'C'], 3, ['A', 'B', 'C'])
-------------
-------------
get_subsequences(['A', 'B', 'C'], 3, ['A', 'C'])
-------------
-------------
get_subsequences(['A', 'B', 'C'], 2, ['B'])
get_subsequences(['A', 'B', 'C'], 3, ['B', 'C'])
-------------
-------------
get_subsequences(['A', 'B', 'C'], 3, ['C'])
-------------


['A', 'A,B', 'A,C', 'B', 'B,C', 'C']

## fp-growth算法
### 了解就好,就是apriori的优化方案, 后续只要会用fp-growth就可以
### 本质就是用树来存储事务,只需两次遍历即可计算出频繁项,性能大大提高!!!

In [175]:

# 2. 加载数据
transactions = []
with open("./retail.txt", "r") as f:
    for line in f:
        items = list(map(int, line.strip().split()))
        transactions.append(items)

print(f"交易数: {len(transactions)}")
print(f"示例交易: {transactions[0]}")

# ⚠️ 采样（可选），否则数据太大挖掘会比较慢
sample_transactions = transactions[:10000]

# 3. 转换为 One-Hot 矩阵
te = TransactionEncoder()
te_ary = te.fit(sample_transactions).transform(sample_transactions)
df = pd.DataFrame(te_ary, columns=te.columns_)

print("One-Hot 矩阵维度:", df.shape)
print("One-Hot values:", df.head(10))


交易数: 88162
示例交易: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
One-Hot 矩阵维度: (10000, 8600)
One-Hot values:     1      2      3      4      5      6      7      8      9      10    ...  \
0   True   True   True   True   True   True   True   True   True   True  ...   
1  False  False  False  False  False  False  False  False  False  False  ...   
2  False  False  False  False  False  False  False  False  False  False  ...   
3  False  False  False  False  False  False  False  False  False  False  ...   
4  False  False  False  False  False  False  False  False  False  False  ...   
5  False  False  False  False  False  False  False  False  False  False  ...   
6  False  False  False  False  False  False  False  False  False  False  ...   
7  False  False  False   True  False  False  False  False  False  False  ...   
8  False  False  False  False  False  False  False  False  False  False  ...   
9  False  False  False  Fal

In [176]:

# 4. FP-Growth 挖掘频繁项集 -- 这个可以计算出retail的频繁项集,但apriori是计算不出来的,超时
frequent_itemsets = fpgrowth(df, min_support=0.02, use_colnames=True)  # 支持度阈值可调整
print("\n频繁项集示例:")
print(frequent_itemsets.head())



频繁项集示例:
   support itemsets
0   0.1828     (33)
1   0.5489     (40)
2   0.2663     (42)
3   0.1722     (39)
4   0.0321     (37)


In [178]:

# 5. 生成关联规则
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
print("\n关联规则示例:")
rules.head()


关联规则示例:


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(33),(42),0.1828,0.2663,0.0597,0.326586,1.226385,1.0,0.011020,1.089524,0.225888,0.153313,0.082168,0.275385
1,(42),(33),0.2663,0.1828,0.0597,0.224183,1.226385,1.0,0.011020,1.053342,0.251596,0.153313,0.050640,0.275385
2,(33),(49),0.1828,0.4312,0.0947,0.518053,1.201420,1.0,0.015877,1.180212,0.205154,0.182361,0.152694,0.368836
3,(49),(33),0.4312,0.1828,0.0947,0.219620,1.201420,1.0,0.015877,1.047182,0.294747,0.182361,0.045056,0.368836
4,"(40, 33)",(42),0.1003,0.2663,0.0427,0.425723,1.598659,1.0,0.015990,1.277606,0.416223,0.131831,0.217286,0.293034
